In [1]:
# Load the API key from the .env file (never hardcode keys!)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Check your .env file.")

client = OpenAI(api_key=api_key)
print("✓ OpenAI client initialized")
print(f"Key loaded: {api_key[:7]}...")  # only show the prefix, never the full key

# Checkpoint: list the models this key can access
models = [m.id for m in client.models.list()]
print(f"✓ Connection works – {len(models)} models available")
print([m for m in models if "gpt" in m][:15])

✓ OpenAI client initialized
Key loaded: sk-svca...
✓ Connection works – 98 models available
['gpt-5.4-2026-03-05', 'gpt-5-nano', 'gpt-5.4-mini', 'gpt-realtime-2.1-mini', 'gpt-realtime-2025-08-28', 'gpt-5-mini', 'gpt-5-mini-2025-08-07', 'gpt-realtime-1.5', 'gpt-realtime', 'gpt-5.2-pro', 'gpt-5.5-pro', 'gpt-audio-2025-08-28', 'gpt-4o-mini-transcribe-2025-03-20', 'gpt-live-transcribe', 'gpt-5.6-luna']


In [2]:
# Debug: check the key for characters that are not allowed in HTTP headers
print("Length:", len(api_key))
print("Contains whitespace/line break:", any(c.isspace() for c in api_key))
print("Non-ASCII characters:", [c for c in api_key if ord(c) > 127])
print("Contains quotes:", any(q in api_key for q in ['"', "'"]))

Length: 167
Contains whitespace/line break: False
Non-ASCII characters: []
Contains quotes: False


In [3]:
# Show position and type of whitespace characters in the key (key itself stays hidden)
print([(i, repr(c)) for i, c in enumerate(api_key) if c.isspace()])

[]


In [4]:
# Install: pip install datasets
%pip install datasets requests pillow pandas openai python-dotenv
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Note: you may need to restart the kernel to use updated packages.


/opt/anaconda3/envs/ironhack/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [5]:
# Work with a smaller sample to keep API costs low
sample_df = products_df.head(20).copy()

# The 'image' column holds PIL objects, not file paths -> save them to disk
image_paths = []
for _, row in sample_df.iterrows():
    path = images_dir / f"{row['id']}.jpg"
    row["image"].convert("RGB").save(path, "JPEG")
    image_paths.append(str(path))
sample_df["image_path"] = image_paths

# Rename columns to match what the prompt function expects
sample_df = sample_df.rename(columns={"productDisplayName": "name",
                                      "masterCategory": "category"})

print(sample_df[["id", "name", "category", "articleType", "baseColour", "image_path"]].head())

      id                                           name     category  \
0  15970               Turtle Check Men Navy Blue Shirt      Apparel   
1  39386             Peter England Men Party Blue Jeans      Apparel   
2  59263                       Titan Women Silver Watch  Accessories   
3  21379  Manchester United Men Solid Black Track Pants      Apparel   
4  53759                          Puma Men Grey T-shirt      Apparel   

   articleType baseColour                image_path  
0       Shirts  Navy Blue  product_images/15970.jpg  
1        Jeans       Blue  product_images/39386.jpg  
2      Watches     Silver  product_images/59263.jpg  
3  Track Pants      Black  product_images/21379.jpg  
4      Tshirts       Grey  product_images/53759.jpg  


In [6]:
# Dataset has no prices -> assign placeholder demo prices by article type
# (NOTE: these are illustrative values, not real prices)
demo_prices = {
    "Shirts": 29.99, "Tshirts": 19.99, "Tops": 24.99, "Kurtas": 34.99,
    "Jeans": 49.99, "Track Pants": 29.99, "Shorts": 24.99, "Skirts": 29.99,
    "Dresses": 44.99, "Sweatshirts": 39.99, "Jackets": 69.99,
    "Casual Shoes": 59.99, "Sports Shoes": 69.99, "Formal Shoes": 79.99,
    "Sandals": 29.99, "Flip Flops": 14.99, "Heels": 54.99, "Flats": 34.99,
    "Watches": 89.99, "Handbags": 59.99, "Wallets": 24.99, "Belts": 19.99,
    "Sunglasses": 39.99, "Socks": 9.99, "Caps": 14.99, "Backpacks": 44.99,
}
DEFAULT_PRICE = 29.99

sample_df["price"] = sample_df["articleType"].map(demo_prices).fillna(DEFAULT_PRICE)

# Check which article types fell back to the default price
missing = sample_df.loc[~sample_df["articleType"].isin(demo_prices), "articleType"].unique()
print("Using default price for:", list(missing) if len(missing) else "none")

print(sample_df[["name", "articleType", "price", "image_path"]])

Using default price for: ['Bra']
                                               name   articleType  price  \
0                  Turtle Check Men Navy Blue Shirt        Shirts  29.99   
1                Peter England Men Party Blue Jeans         Jeans  49.99   
2                          Titan Women Silver Watch       Watches  89.99   
3     Manchester United Men Solid Black Track Pants   Track Pants  29.99   
4                             Puma Men Grey T-shirt       Tshirts  19.99   
5              Inkfruit Mens Chain Reaction T-shirt       Tshirts  19.99   
6                  Fabindia Men Striped Green Shirt        Shirts  29.99   
7                     Jealous 21 Women Purple Shirt        Shirts  29.99   
8                          Puma Men Pack of 3 Socks         Socks   9.99   
9                            Skagen Men Black Watch       Watches  89.99   
10  Puma Men Future Cat Remix SF Black Casual Shoes  Casual Shoes  59.99   
11           Fossil Women Black Huarache Weave Belt    

In [8]:
import base64

def encode_image_to_base64(image_path):
    """Encode an image file to a base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded

# Example usage (template used products_df, but image_path only exists in sample_df)
sample_path = sample_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)

print(f"Image: {sample_path}")
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Image: product_images/15970.jpg
Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [10]:
# Choose the model once, so it's easy to change later
MODEL = "gpt-5-mini"

def build_additional_info(row):
    """Combine extra metadata from the dataset into one string for the prompt."""
    return (f"Article type: {row['articleType']}, colour: {row['baseColour']}, "
            f"target group: {row['gender']}, usage: {row['usage']}, season: {row['season']}")


def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """Create a prompt for generating product listings."""
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}

Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    return prompt
# Test the prompt with the first product of our sample
first = sample_df.iloc[0]
test_prompt = create_product_listing_prompt(
    product_name=first["name"],
    price=first["price"],
    category=first["category"],
    additional_info=build_additional_info(first),
)

print(test_prompt)

You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Turtle Check Men Navy Blue Shirt
- Price: $29.99
- Category: Apparel
- Additional Info: Article type: Shirts, colour: Navy Blue, target group: Men, usage: Casual, season: Fall

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}

Be specific about what you see in the imag

In [11]:
import json

def generate_listing(row, model=MODEL):
    """Send image + prompt for one product to the API and return the parsed listing."""
    image_b64 = encode_image_to_base64(row["image_path"])
    prompt = create_product_listing_prompt(
        product_name=row["name"],
        price=row["price"],
        category=row["category"],
        additional_info=build_additional_info(row),
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
            ],
        }],
        response_format={"type": "json_object"},  # ask the API to return valid JSON
    )

    raw_text = response.choices[0].message.content
    listing = json.loads(raw_text)  # turn the JSON text into a Python dictionary
    return listing, response

# Test with the first product
listing, response = generate_listing(sample_df.iloc[0])

print("TITLE:", listing["title"])
print("\nDESCRIPTION:", listing["description"])
print("\nFEATURES:")
for f in listing["features"]:
    print(" -", f)
print("\nKEYWORDS:", listing["keywords"])
print("\nTokens used:", response.usage.total_tokens)

TITLE: Turtle Check Men's Navy Blue Casual Shirt

DESCRIPTION: Refresh your fall wardrobe with the Turtle Check Men’s Navy Blue Shirt — a modern, versatile top built for everyday comfort and style. The shirt showcases a crisp navy-blue and white check pattern that reads classic but contemporary. Visible details include a neat spread collar, full button-front, a single chest pocket, long sleeves with buttoned cuffs and a slightly curved hem that looks great tucked or untucked. The tailored silhouette on the model gives a streamlined, flattering fit without feeling restrictive. Crafted from a lightweight woven fabric with a soft, cotton-like hand, it layers easily under jackets or wears solo with jeans for casual outings. Practical stitching and durable buttons ensure long-lasting wear, while the color and pattern make it an easy match for chinos, denim, or layered fall looks. At $29.99, this shirt delivers everyday polish and comfort—perfect for weekend plans, casual office days, or sea

In [12]:
# Save the first result (old prompt) for a before/after comparison
listing_before = listing

# Improved prompt v2: the original prompt + an extra rule against invented details
def create_product_listing_prompt_v2(product_name, price, category, additional_info=None):
    base_prompt = create_product_listing_prompt(product_name, price, category, additional_info)
    extra_rule = ("\nOnly describe details that are clearly visible in the image or given in the "
                  "product information. Do not invent materials, fabric feel, pockets or other "
                  "details you cannot verify.")
    return base_prompt + extra_rule

# Check: show the end of the new prompt
print("Saved old result:", listing_before["title"])
print("\nEnd of new prompt:\n", create_product_listing_prompt_v2("Test", 9.99, "Test")[-300:])

Saved old result: Turtle Check Men's Navy Blue Casual Shirt

End of new prompt:
 }

Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features.
Only describe details that are clearly visible in the image or given in the product information. Do not invent materials, fabric feel, pockets or other details you cannot verify.


In [13]:
# Same as generate_listing, but using the improved prompt v2
def generate_listing_v2(row, model=MODEL):
    image_b64 = encode_image_to_base64(row["image_path"])
    prompt = create_product_listing_prompt_v2(
        product_name=row["name"],
        price=row["price"],
        category=row["category"],
        additional_info=build_additional_info(row),
    )
    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
            ],
        }],
        response_format={"type": "json_object"},
    )
    raw_text = response.choices[0].message.content
    return json.loads(raw_text), response

# Run product 1 again with the new prompt
listing_after, response_after = generate_listing_v2(sample_df.iloc[0])

# Raw JSON exactly as returned by the API (evidence for lab_proof.md)
print("RAW JSON OUTPUT (new prompt):")
print(response_after.choices[0].message.content)

# Before/after comparison of the features
print("\n" + "=" * 50)
print("FEATURES BEFORE (old prompt):")
for f in listing_before["features"]:
    print(" -", f)
print("\nFEATURES AFTER (new prompt):")
for f in listing_after["features"]:
    print(" -", f)

RAW JSON OUTPUT (new prompt):
{
  "title": "Turtle Check Men's Navy Blue Checked Shirt",
  "description": "A timeless casual essential, the Turtle Check Men Navy Blue Shirt pairs classic checks with everyday versatility. The shirt features a navy-blue base with crisp white and light-blue grid checks for a clean, contemporary look. Visible details include a classic collar, full button-front closure and a single chest pocket — all shown on a modern, streamlined silhouette. The sleeves are shown rolled for a relaxed, casual vibe, making this shirt ideal for layered fall outfits or weekend wear. Thoughtfully designed to be easy to style, it looks great over a tee or tucked into chinos for smart-casual occasions. The balanced check pattern and rich navy tones bring instant polish to your seasonal wardrobe while keeping the look approachable and comfortable. Whether you’re dressing for a casual office, coffee with friends, or a weekend outing, this shirt delivers an understated, versatile st

In [14]:
def process_products(df):
    """Generate listings for all products in df. Errors are caught per product."""
    results = []
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        print(f"[{i}/{len(df)}] {row['name']}", end=" ... ")
        try:
            listing, response = generate_listing_v2(row)
            results.append({
                "id": row["id"],
                "name": row["name"],
                "price": row["price"],
                "status": "success",
                **listing,  # adds title, description, features, keywords
                "tokens": response.usage.total_tokens,
            })
            print("✓")
        except Exception as e:
            # One failing product should not stop the whole batch
            results.append({
                "id": row["id"],
                "name": row["name"],
                "price": row["price"],
                "status": "error",
                "error": f"{type(e).__name__}: {e}",
            })
            print(f"✗ {type(e).__name__}")
    return results

# Test run with only 3 products first
test_results = process_products(sample_df.head(3))

[1/3] Turtle Check Men Navy Blue Shirt ... ✓
[2/3] Peter England Men Party Blue Jeans ... ✓
[3/3] Titan Women Silver Watch ... ✓


In [15]:
import time

start = time.time()
all_results = process_products(sample_df)
duration = time.time() - start

# Summary
successes = [r for r in all_results if r["status"] == "success"]
errors = [r for r in all_results if r["status"] == "error"]
total_tokens = sum(r["tokens"] for r in successes)

print("\n" + "=" * 50)
print(f"Processed: {len(all_results)} products in {duration:.0f} seconds")
print(f"Success: {len(successes)} | Errors: {len(errors)}")
print(f"Total tokens: {total_tokens} (avg {total_tokens / max(len(successes), 1):.0f} per product)")

[1/20] Turtle Check Men Navy Blue Shirt ... ✓
[2/20] Peter England Men Party Blue Jeans ... ✓
[3/20] Titan Women Silver Watch ... ✓
[4/20] Manchester United Men Solid Black Track Pants ... ✓
[5/20] Puma Men Grey T-shirt ... ✓
[6/20] Inkfruit Mens Chain Reaction T-shirt ... ✓
[7/20] Fabindia Men Striped Green Shirt ... ✓
[8/20] Jealous 21 Women Purple Shirt ... ✓
[9/20] Puma Men Pack of 3 Socks ... ✓
[10/20] Skagen Men Black Watch ... ✓
[11/20] Puma Men Future Cat Remix SF Black Casual Shoes ... ✓
[12/20] Fossil Women Black Huarache Weave Belt ... ✓
[13/20] Fila Men Cush Flex Black Slippers ... ✓
[14/20] Murcia Women Blue Handbag ... ✓
[15/20] Ben 10 Boys Navy Blue Slippers ... ✓
[16/20] Reid & Taylor Men Check Purple Shirts ... ✓
[17/20] Police Men Black Dial Watch PL12889JVSB ... ✓
[18/20] Gini and Jony Girls Knit White Top ... ✓
[19/20] Bwitch Beige Full-Coverage Bra BW335 ... ✓
[20/20] Baggit Women Brown Handbag ... ✓

Processed: 20 products in 307 seconds
Success: 20 | Errors: 0
To

In [16]:
import json

# Save all generated listings as JSON
with open("generated_listings.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

print(f"✓ Saved {len(all_results)} listings to generated_listings.json")

✓ Saved 20 listings to generated_listings.json


In [17]:
# Demonstrate graceful error handling:
# product 1 has a missing image file, product 2 is fine
demo_df = sample_df.head(2).copy()
demo_df.loc[demo_df.index[0], "image_path"] = "product_images/does_not_exist.jpg"
demo_df.loc[demo_df.index[0], "name"] = "BROKEN TEST PRODUCT (missing image)"

error_demo = process_products(demo_df)

print("\nResult per product:")
for r in error_demo:
    print(f" - {r['name']}: {r['status']} {r.get('error', '')}")

[1/2] BROKEN TEST PRODUCT (missing image) ... ✗ FileNotFoundError
[2/2] Peter England Men Party Blue Jeans ... ✓

Result per product:
 - BROKEN TEST PRODUCT (missing image): error FileNotFoundError: [Errno 2] No such file or directory: 'product_images/does_not_exist.jpg'
 - Peter England Men Party Blue Jeans: success 
